# 4.4 混合位置後怎麼處理特徵？


注意力讀了其他位置，回到當前位置時已經帶回一排新特徵。但位置之間的交流與一張卡內如何組合特徵，是兩件不同的事。像先從鄰居收集食材，再在自己的廚房調配；模型通常讓注意力負責前者，再讓每個位置使用同一套配方處理自己手上的數。這套位置內計算叫前饋網路（feed-forward network，FFN）。

本章用兩個Linear夾一個非線性：先把D特徵擴成較多隱藏特徵，經過轉換，再縮回D，才能與 [4.2](https://birdhackor.github.io/tiny-perceptron-vlm/4.2.html) 的主路相加。Linear與非線性的原因見 [2.3](https://birdhackor.github.io/tiny-perceptron-vlm/2.3.html)、[2.4](https://birdhackor.github.io/tiny-perceptron-vlm/2.4.html)。擴張提供更多中間組合，不是創造更多文字位置；序列長度從頭到尾不改。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from tiny_perceptron.modern import DenseFFN

torch.manual_seed(42)
layer = DenseFFN(4)
x = torch.ones(1, 2, 4)
y = layer(x)
print("擴張權重", layer.up.weight.shape)
print("縮回權重", layer.down.weight.shape)
print(y.shape, y)
assert torch.allclose(y[:, 0], y[:, 1])

`torch.manual_seed(42)`固定亂數種子，讓這次權重初始化與結果可以重現；42只是本次選定的起點，沒有特殊模型意義。輸入一筆、兩位置、每位置四個1，形狀 `[1,2,4]`。`DenseFFN(4)` 預設先擴到16，再縮回4，兩張權重分別為 `[16,4]` 與 `[4,16]`。每位置依序做四到十六、非線性、十六到四，輸出仍 `[1,2,4]`。兩個位置輸入完全相同，輸出兩排也相同；比較的是具體四個數逐格一致，不是只比較shape。

這個工具預設的非線性叫GELU，它平滑地按輸入大小調整數值，較負的輸入被縮小、較正的通常保留更多；與ReLU直接把負數截成零不同，GELU會保留部分小負值。本節重點是非線性夾在兩個配方之間，暫時不必記完整公式。權重與偏移都從亂數開始，所以數值代表這次初始化，不表示兩張全1卡有固定語意。

Dense的意思是每個位置都使用這套完整FFN參數，不依當前字選擇只用某部分配方。這是本章的簡單起點，後續會介紹選擇專家配方的方法；現在先確認各位置共用參數、獨立處理特徵。共享不表示位置相互讀取：改第1位置的輸入，不會通過這層FFN改變第0位置，跨位置依賴已經在注意力層完成。

數學上可寫「擴張Linear→GELU→縮回Linear」，每一步都作用在最後特徵軸，其餘軸保留。若兩位置的表示在注意力後不同，FFN也可能給不同結果，即使使用同樣配方；它不負責讓所有位置統一成同一張卡。

練習保存y後把 `x[0,1,0]` 改成2，再算一次 `changed=layer(x)`。先預測 `changed[:,0]` 與 `y[:,0]` 相同，而第二位置通常不同，再用 `torch.allclose` 核對。這次明確只改某個位置一格，讓你看見位置內計算和跨位置混合的界線。
